# Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
import joblib

# Loading of processed data

In [2]:
df = pd.read_parquet('../data/processed/backrooms_processed.parquet')

X = df.drop(['survived_24h', 'escape_probability', 'survival_time_hours'], axis=1)
y = df['survived_24h']

# Train, Val Splitting

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y) # train
X_test.to_parquet('../data/processed/X_test.parquet', index=False)
y_test.to_frame().to_parquet('../data/processed/y_test.parquet', index=False)

# Baseline Model


In [4]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print('Baseline score: ', model.score(X_test, y_test))

Baseline score:  0.6778


/Users/zimbazo/Documents/CodeProjects/BackroomsML/venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Advanced Models + Tuning

## Model dict

In [5]:
# models = {
#     'LogisticRegression': LogisticRegression(),
#     'RandomForestClassifier': RandomForestClassifier(),
#     'GradientBoostingClassifier': GradientBoostingClassifier(),
#     'XGBoost': xgb.XGBClassifier(

#     ),
#     'LightGBM': lgb.LGBMClassifier(
        
#     )
# }

In [ ]:
rf = RandomForestClassifier(random_state=42)
# rf = GradientBoostingClassifier(random_state=42)

param_grid = {
    'n_estimators': [250],
    'max_depth': [8],
    # 'n_estimators': [100],
    # 'learning_rate': [0.01, 0.1, 0.2],
    # 'max_depth': [3, 5]
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring='roc_auc')
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

In [ ]:
print(f'score: {best_model.score(X_test, y_test)}')
print(f'params: {best_model.get_params()}')

/Users/zimbazo/Documents/CodeProjects/BackroomsML/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.

# Model saving

In [8]:
joblib.dump(best_model, '../models/best_model_v1.pkl')
print('best model has been saved to /models')

best model has been saved to /models
